# Merge Notebook for Waiver Level and Service Level Data

# Waiver Level Merge Script

## Merge Top + Secondary + Tertiary Extractions

Combines the three per-tier merged CSVs (already HTML+text merged) into one
flat dataset keyed by normalized `document_id`.

**Merge order:**
1. Load `merged_top.csv`, `merged_secondary.csv`, `merged_tertiary.csv`
2. Normalize doc IDs (remove spaces/dots/dashes, uppercase)
3. Outer-join all three on `document_id`
4. Save as `output/merged_all.csv`

*(PDF AcroForm merge is in Section 5 — run separately once ready.)*

In [10]:
import sys
from pathlib import Path
import os, csv

sys.path.insert(0, str(Path.cwd().parent))

from merge.merge_extractions import normalize_doc_id, is_valid_waiver_id, is_empty, compute_fill_rate

import pandas as pd
import numpy as np

In [3]:
# ── Configuration ──
TOP_CSV       = '../output/merged_top.csv'
SECONDARY_CSV = '../output/merged_secondary.csv'
TERTIARY_CSV  = '../output/merged_tertiary.csv'
OUTPUT_CSV    = '../output/merged_all.csv'

PDF_CSV       = '../output/pdf_acroform_extraction.csv'  # used in Section 5

## 1. Load tier CSVs

In [4]:
df_top  = pd.read_csv(TOP_CSV)
df_sec  = pd.read_csv(SECONDARY_CSV)
df_ter  = pd.read_csv(TERTIARY_CSV)

print(f'Top:       {df_top.shape}')
print(f'Secondary: {df_sec.shape}')
print(f'Tertiary:  {df_ter.shape}')

Top:       (1151, 81)
Secondary: (1151, 27)
Tertiary:  (1178, 60)


## 2. Normalize doc IDs and filter invalid rows

In [5]:
def prep(df, label):
    df = df.copy()
    df['document_id'] = df['document_id'].apply(normalize_doc_id)
    before = len(df)
    df = df[df['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
    df = df.drop_duplicates(subset=['document_id'], keep='first')
    print(f'{label}: {before} → {len(df)} rows after filtering/dedup')
    return df

df_top = prep(df_top, 'Top')
df_sec = prep(df_sec, 'Secondary')
df_ter = prep(df_ter, 'Tertiary')

Top: 1151 → 1151 rows after filtering/dedup
Secondary: 1151 → 1151 rows after filtering/dedup
Tertiary: 1178 → 1178 rows after filtering/dedup


## 3. Outer-join all three tiers on document_id

In [6]:
merged = (
    df_top
    .merge(df_sec, on='document_id', how='outer', suffixes=('', '_sec'))
    .merge(df_ter, on='document_id', how='outer', suffixes=('', '_ter'))
)

# Replace NaN strings with empty string for consistency
merged = merged.fillna('')

print(f'Merged shape: {merged.shape}')
print(f'Unique doc IDs: {merged["document_id"].nunique()}')
display(merged.head(5))

Merged shape: (1178, 166)
Unique doc IDs: 1178


,document_id,title,waiver_type,effective_date,hospital_loc,hospital_loc_limits,nursing_facility_loc,nursing_facility_loc_limits,ifc_loc,ifc_loc_limits,...,transition_plan_1,transition_plan_2,transition_plan_3,transition_plan_4,transition_plan_5,transition_plan_6,transition_plan_7,transition_plan_8,transition_plan_9,transition_plan_10
0,AK0260R0600,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/21,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AK0260R0602,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/22,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AK0260R0604,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/23,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AK0260R0607,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/24,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AK0261R0301,Older Alaskans renewal waiver,Regular Waiver,07/01/08,0.0,,1.0,,0.0,,...,,,,,,,,,,


## 4. Save merged_all.csv

In [ ]:
import os, csv

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved {len(merged)} records → {OUTPUT_CSV}')

In [7]:
PDF_CSV

'../output/pdf_acroform_extraction.csv'

## 5. (Later) Merge with PDF AcroForm

Run this section once `pdf_acroform_extraction.csv` is ready.
PDF is authoritative for radio-button fields (those values are read directly
from the form state and are more reliable than text parsing).

Fields marked authoritative come from the PDF when available; all other fields
fall back to the HTML+text merged value.

In [11]:
from merge.merge_extractions import merge_two_sources

# Fields where PDF AcroForm overrides HTML+text (radio buttons only)
PDF_AUTHORITATIVE = [
    'approval_period',
    'waive_1902a',
    'waive_statewideness',
    'statecontracts_mcos1', 'statecontracts_mcos2',
    'statecontracts_mcos3', 'statecontracts_mcos4',
    'payforresidential',
    'reimburse_paidcg',
]

if os.path.exists(PDF_CSV):
    df_pdf = pd.read_csv(PDF_CSV)
    df_pdf['document_id'] = df_pdf['document_id'].apply(normalize_doc_id)
    df_pdf = df_pdf[df_pdf['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
    print(f'PDF AcroForm: {df_pdf.shape}')

    merged_final = merge_two_sources(
        merged, df_pdf,
        source_a_name='top+sec+ter',
        source_b_name='pdf_acroform',
        authoritative_fields=PDF_AUTHORITATIVE,
        authoritative_source='b',
    )

    FINAL_CSV = OUTPUT_CSV.replace('merged_all', 'merged_all_with_pdf')
    merged_final.to_csv(FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)
    print(f'Saved {len(merged_final)} records → {FINAL_CSV}')
else:
    print(f'PDF CSV not found at {PDF_CSV} — skipping.')

PDF AcroForm: (779, 20)
Source top+sec+ter: 1178 unique IDs
Source pdf_acroform: 779 unique IDs
  Only in top+sec+ter: 442
  Only in pdf_acroform: 43
  Overlap: 736

Merged: 1221 rows, 185 columns
Saved 1221 records → ../output/merged_all_with_pdf.csv


In [12]:
import os, csv

PDF_FINAL_CSV = "../output/merged_all_with_pdf.csv"

df_pdf = pd.read_csv(PDF_CSV)
df_pdf["document_id"] = df_pdf["document_id"].apply(normalize_doc_id)
df_pdf = df_pdf[df_pdf["document_id"].apply(is_valid_waiver_id)].reset_index(drop=True)
df_pdf = df_pdf.drop_duplicates(subset=["document_id"], keep="first")
print(f"PDF AcroForm: {df_pdf.shape}")

merged_with_pdf = merged.merge(df_pdf, on="document_id", how="left")
merged_with_pdf = merged_with_pdf.fillna("")

# ── Column order ──────────────────────────────────────────────────────────────
FINAL_COLUMN_ORDER = [
    # Key
    "document_id",
    # Waiver overview
    "title",
    "waiver_type",
    "effective_date",
    "approval_period",
    # Level of care
    "hospital_loc",
    "hospital_loc_limits",
    "nursing_facility_loc",
    "nursing_facility_loc_limits",
    "ifc_loc",
    "ifc_loc_limits",
    "local_eval",
    "local_eval_instrument",
    "reeval_sched",
    # Concurrent waivers
    "concurrent_1915a",
    "concurrent_1915b",
    "concurrent_1932a",
    "concurrent_1915i",
    "concurrent_1915j",
    "concurrent_1115",
    # Eligibility — geography & groups
    "dual_elg",
    "waive_geographic_limits",
    "waive_geographic_lipd",
    "aged_group",
    "aged_group_min",
    "aged_group_max",
    "physicaldis_group",
    "physicaldis_group_min",
    "physicaldis_group_max",
    "otherdis_group",
    "otherdis_group_min",
    "otherdis_group_max",
    "braininjury_group",
    "braininjury_group_min",
    "braininjury_group_max",
    "hivaids_group",
    "hivaids_group_min",
    "hivaids_group_max",
    "medicallyfrail_group",
    "medicallyfrail_group_min",
    "medicallyfrail_group_max",
    "techdep_group",
    "techdep_group_min",
    "techdep_group_max",
    "autism_group",
    "autism_group_min",
    "autism_group_max",
    "dd_group",
    "dd_group_min",
    "dd_group_max",
    "id_group",
    "id_group_min",
    "id_group_max",
    "mi_group",
    "mi_group_min",
    "mi_group_max",
    "sed_group",
    "sed_group_min",
    "sed_group_max",
    # Eligibility — criteria
    "eligibility_1",
    "eligibility_2",
    "eligibility_3",
    "eligibility_4",
    "eligibility_5",
    "eligibility_5_percent",
    "eligibility_5_100",
    "eligibility_6",
    "eligibility_7",
    "eligibility_8",
    "eligibility_9",
    "eligibility_10",
    "eligibility_11",
    "eligibility_12",
    "spousal_impov_a",
    "spousal_impov_bc",
    # Cost / enrollment
    "cost_limit_pcntaboveinstit",
    "costlimit",
    "numberofbenes_year1",
    "numberofbenes_year2",
    "numberofbenes_year3",
    "numberofbenes_year4",
    "numberofbenes_year5",
    "max_numberofbenes_year1",
    "max_numberofbenes_year2",
    "max_numberofbenes_year3",
    "max_numberofbenes_year4",
    "max_numberofbenes_year5",
    "numberbenes_limited",
    "phaseinoutschedule",
    "entrantselection",
    "specialHCBS",
    "enhanced_payments_yes",
    "statecontracts_mcos",
    # Self-direction (Appendix E)
    "selfdirection_yes",
    "selfdirection_description",
    "sd_authority",
    "sd_election",
    "sd_livarrngmt_1",
    "sd_livarrngmt_2",
    "sd_livarrngmt_3",
    "sd_service_1",
    "sd_service_1_ea",
    "sd_service_1_ba",
    "sd_fms_gov",
    "sd_fms_pe",
    "scope_fms_1",
    "scope_fms_2",
    "scope_fms_3",
    "scope_fms_4",
    "sd_numenrollees_ea1",
    "sd_numenrollees_ea2",
    "sd_numenrollees_ea3",
    "sd_numenrollees_ea4",
    "sd_numenrollees_ea5",
    "sd_numenrollees_ba1",
    "sd_numenrollees_ba2",
    "sd_numenrollees_ba3",
    "sd_numenrollees_ba4",
    "sd_numenrollees_ba5",
    "sd_coemployer",
    "sd_commonlaw",
    "provider_rate_methods",
    "payforresidential",
    "reimburse_paidcg",
    # Waiver services (Tertiary)
    "ma_1",
    "ma_2",
    "ma_3",
    "ma_4",
    "ma_5",
    "ma_6",
    "ma_7",
    "ma_8",
    "ma_9",
    "ma_10",
    "ma_11",
    "ma_12",
    "osa_1",
    "osa_2",
    "osa_3",
    "osa_4",
    "osa_5",
    "osa_6",
    "osa_7",
    "osa_8",
    "osa_9",
    "osa_10",
    "osa_11",
    "osa_12",
    "ce_1",
    "ce_2",
    "ce_3",
    "ce_4",
    "ce_5",
    "ce_6",
    "ce_7",
    "ce_8",
    "ce_9",
    "ce_10",
    "ce_11",
    "ce_12",
    "inse_1",
    "inse_2",
    "inse_3",
    "inse_4",
    "inse_5",
    "inse_6",
    "inse_7",
    "inse_8",
    "inse_9",
    "inse_10",
    "inse_11",
    "inse_12",
    # Descriptions / Transition
    "waiver_description",
    "transitionplan_1",
    "transitionplan_2",
    "transitionplan_3",
    "transitionplan_4",
    "transitionplan_5",
    "transitionplan_6",
    "transitionplan_7",
    "transitionplan_8",
    "transitionplan_9",
    "transitionplan_10",
]

# Keep only columns that actually exist (guards against future schema changes)
final_cols = [c for c in FINAL_COLUMN_ORDER if c in merged_with_pdf.columns]
missing = [c for c in FINAL_COLUMN_ORDER if c not in merged_with_pdf.columns]
if missing:
    print(f"Warning — columns in order list but not in data: {missing}")

merged_with_pdf = merged_with_pdf[final_cols]

print(f"Final shape: {merged_with_pdf.shape}")
merged_with_pdf.to_csv(PDF_FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved {len(merged_with_pdf)} records → {PDF_FINAL_CSV}")

display(merged_with_pdf.head(3))

PDF AcroForm: (779, 20)
Final shape: (1178, 183)
Saved 1178 records → ../output/merged_all_with_pdf.csv


,document_id,title,waiver_type,effective_date,approval_period,hospital_loc,hospital_loc_limits,nursing_facility_loc,nursing_facility_loc_limits,ifc_loc,...,transition_plan_1,transition_plan_2,transition_plan_3,transition_plan_4,transition_plan_5,transition_plan_6,transition_plan_7,transition_plan_8,transition_plan_9,transition_plan_10
0,AK0260R0600,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/21,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AK0260R0602,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/22,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AK0260R0604,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/23,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# merge the misc Vs merged all(text+html+acroform)
# Set KEEP_MISC_ONLY = True to also append misc-only doc IDs

KEEP_MISC_ONLY = True  # ← change to True to include misc-only rows

import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

def clean_id(v):
    if pd.isna(v): return ''
    return str(v).strip()

base = pd.read_csv('../output/merged_all_with_pdf.csv')
misc = pd.read_csv('../output/misc_extraction2.csv')

base['document_id'] = base['document_id'].apply(clean_id)
misc['document_id'] = misc['document_id'].apply(clean_id)

ids_base = set(base['document_id'])

# Fill empty cells for matching doc IDs
misc_lookup = misc.set_index('document_id').to_dict('index')
shared_cols = [c for c in misc.columns if c in base.columns and c != 'document_id']

filled = 0
for idx, row in base.iterrows():
    doc_id = row['document_id']
    misc_row = misc_lookup.get(doc_id, {})
    for col in shared_cols:
        if is_empty(row[col]) and not is_empty(misc_row.get(col)):
            base.at[idx, col] = misc_row[col]
            filled += 1

if KEEP_MISC_ONLY:
    only_misc = misc[~misc['document_id'].isin(ids_base)].copy()
    print(f'Appending {len(only_misc)} misc-only rows')
    base = pd.concat([base, only_misc], ignore_index=True)
else:
    print('misc-only rows dropped (KEEP_MISC_ONLY=False)')

base = base.sort_values('document_id').reset_index(drop=True)
base.to_csv('../output/merged_all_with_pdf_updated.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Cells filled from misc: {filled}')
print(f'Final shape: {base.shape}')


/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  base.at[idx, col] = misc_row[col]
/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  base.at[idx, col] = misc_row[col]
/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dt

Appending 39 misc-only rows
Cells filled from misc: 17429
Final shape: (1217, 186)


In [27]:
# applying athe clean text pipeline ( reomve url, page numbers, etc) to the merged_all_with_pdf_updated.csv output from above, and save back to same file (overwriting it)
import pandas as pd, csv, sys
import importlib, sys

# Remove cached module so changes are picked up without restarting kernel
if 'merge.merge_extractions' in sys.modules:
    importlib.reload(sys.modules['merge.merge_extractions'])
from merge.merge_extractions import clean_text_value


df = pd.read_csv('../output/merged_all_with_pdf_updated.csv')
str_cols = [c for c in df.columns if df[c].dtype == object]
for col in str_cols:
    df[col] = df[col].apply(clean_text_value)

df.to_csv('../output/merged_all_with_pdf_updated.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Cleaned {len(str_cols)} string columns. Shape: {df.shape}')

Cleaned 60 string columns. Shape: (1217, 186)


In [28]:
# drop non values 

import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

df = pd.read_csv('../output/merged_all_with_pdf_updated.csv')

# Drop rows with no document_id
before_id = len(df)
df = df[df['document_id'].notna() & (df['document_id'].str.strip() != '')]
print(f'Dropped no doc_id: {before_id - len(df)}')

# Filter rows with >= 80% missing
data_cols = [c for c in df.columns if c != 'document_id']
df['missing_pct'] = df[data_cols].apply(
    lambda row: row.apply(is_empty).sum() / len(data_cols) * 100, axis=1
).round(1)

before = len(df)
df_clean = df[df['missing_pct'] < 80].drop(columns='missing_pct')
dropped  = before - len(df_clean)

df_clean.to_csv('../output/merged_all_with_pdf_drop_nan.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Before: {before} | Dropped (>=80% missing): {dropped} | Remaining: {len(df_clean)}')


Dropped no doc_id: 0
Before: 1217 | Dropped (>=80% missing): 61 | Remaining: 1156


In [29]:
import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

df = pd.read_csv('../output/merged_all_with_pdf_drop_nan.csv')
before = len(df)

# 1. Drop rows with null/empty document_id OR null title
df = df[
    df['document_id'].apply(lambda v: not is_empty(v)) &
    df['title'].apply(lambda v: not is_empty(v))
].reset_index(drop=True)
print(f'Dropped: {before - len(df)} | Remaining: {len(df)}')

# 2. Standardize waiver_type
valid_types = {'Model Waiver', 'Regular Waiver'}
def fix_waiver_type(v):
    if is_empty(v): return 'Regular Waiver'
    s = str(v).strip()
    return s if s in valid_types else 'Regular Waiver'

changed_wt = (df['waiver_type'] != df['waiver_type'].apply(fix_waiver_type)).sum()
df['waiver_type'] = df['waiver_type'].apply(fix_waiver_type)
print(f'waiver_type corrected: {changed_wt}')

# 3. Fill empty approval_period with '5 years'
filled_ap = df['approval_period'].apply(is_empty).sum()
df['approval_period'] = df['approval_period'].apply(
    lambda v: '5 years' if is_empty(v) else v
)
print(f'approval_period filled with "5 years": {filled_ap}')

df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Final shape: {df.shape}')


Dropped: 16 | Remaining: 1140
waiver_type corrected: 17
approval_period filled with "5 years": 411
Final shape: (1140, 186)


In [ ]:
## file based clean up

import re

# Fix min_numservices where section text was captured instead of just the number
# re.match(r'^(\d+)\s*ii\.', x) extracts the leading 1 and returns it. The last case (ii. Frequency... with no leading number) gets blanked. 
def clean_min_numservices(x):
    x = str(x).strip()
    if not x or x in ('', 'nan', 'None'):
        return ''
    # "1ii. Frequency of services..." → "1"
    m = re.match(r'^(\d+)\s*ii\.', x)
    if m:
        return m.group(1)
    # Pure label text with no leading number → blank
    if 'Frequency of services' in x:
        return ''
    return x

if 'min_numservices' in df.columns:
    df['min_numservices'] = df['min_numservices'].apply(clean_min_numservices)
    print('min_numservices cleaned')

# Flag entrantselection rows that captured wrong section text
bad_entrant_patterns = [
    '1. Request Information',
    '3. Nature of the Amendment',
    'Component(s) of the Approved Waiver',
]

flags = []
for idx, row in df.iterrows():
    val = str(row.get('entrantselection', '')).strip()
    if val and val not in ('', 'nan', 'None'):
        if any(p in val for p in bad_entrant_patterns):
            flags.append({
                'document_id': row['document_id'],
                'issue': 'Possible wrong section captured in entrantselection',
                'value': val[:300],
            })

flags_df = pd.DataFrame(flags)
print(f'entrantselection flags: {len(flags_df)}')
display(flags_df)


min_numservices cleaned
entrantselection flags: 7


,document_id,issue,value
0,IL0350R0501,Possible wrong section captured in entrantsele...,3. Nature of the Amendment A. Component(s) of ...
1,LA0121R0504,Possible wrong section captured in entrantsele...,"- Update the ""Adult Day Health Care"" (ADHC) se..."
2,LA0866R0004,Possible wrong section captured in entrantsele...,- Add Housing Stabilization and Housing Transi...
3,LA0866R0200,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
4,OH0337R0500,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
5,TX0221R0700,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
6,UT0292R0600,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...


In [33]:
bad_ids = {
    'IL0350R0501', 'LA0121R0504', 'LA0866R0004',
    'LA0866R0200', 'OH0337R0500', 'TX0221R0700', 'UT0292R0600'
}

mask = df['document_id'].isin(bad_ids)
df.loc[mask, 'entrantselection'] = ''
print(f'Blanked entrantselection for {mask.sum()} rows')


Blanked entrantselection for 7 rows


In [37]:
import re

# 1. Fix § encoding artifact in statecontracts_mcos
if 'statecontracts_mcos' in df.columns:
    df['statecontracts_mcos'] = df['statecontracts_mcos'].str.replace(
        r'\?1115', '§1115', regex=True
    ).str.replace(
        r'\?1915', '§1915', regex=True
    )

# 2. Fix workers' compensation encoding artifact
for col in df.columns:
    df[col] = df[col].astype(str).str.replace(
        r"workersâ\s+compensation", "workers' compensation", regex=True, flags=re.IGNORECASE
    )

# 3. Convert 0.0 / 1.0 → 0 / 1 across all columns
def clean_decimal_int(val):
    if isinstance(val, float) and val == int(val):
        return str(int(val))
    if isinstance(val, str):
        try:
            f = float(val)
            if f == int(f):
                return str(int(f))
        except (ValueError, OverflowError):
            pass
    return val
# Replace all nan strings with empty string
df = df.fillna('').astype(str).replace({'nan': '', 'None': '', '<NA>': ''})

df = df.applymap(clean_decimal_int)

print('Done. Shape:', df.shape)


/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/525681540.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_decimal_int)


Done. Shape: (1140, 186)


In [38]:
import csv

df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Saved. Shape: {df.shape}')


Saved. Shape: (1140, 186)


In [39]:
##### Final saving and converting to dta file

# df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
df.to_stata('../output/1915c-waiver-level-v2.dta', write_index=False, version=118)
df.to_csv('../output/1915c-waiver-level-v2.csv', index=False, quoting=csv.QUOTE_ALL)
print('Saved.')


Saved.


# Service Level Data Merge 

In [20]:
import importlib, sys
if "merge.merge_service_level" in sys.modules:
    importlib.reload(sys.modules["merge.merge_service_level"])
from merge.merge_service_level import clean_service_level_dataframe, merge_service_level

import sys, csv
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from merge.merge_service_level import (
    clean_service_level_dataframe,
    merge_service_level,
)


In [23]:


# Load raw CSVs
df_html = pd.read_csv("../output/html_service_level.csv", dtype=str, low_memory=False)
df_text = pd.read_csv("../output/text_service_level.csv", dtype=str, low_memory=False)

print(f"HTML raw:  {df_html.shape[0]:,} rows x {df_html.shape[1]} cols")
print(f"Text raw:  {df_text.shape[0]:,} rows x {df_text.shape[1]} cols")

HTML raw:  14,782 rows x 29 cols
Text raw:  11,254 rows x 31 cols


In [24]:
# Clean both sources
df_html_clean = clean_service_level_dataframe(df_html, source="html", verbose=True)
print()
df_text_clean = clean_service_level_dataframe(df_text, source="text", verbose=True)

print(f"\nHTML clean: {df_html_clean.shape[0]:,} rows")
print(f"Text clean: {df_text_clean.shape[0]:,} rows")
print("\nReady to merge.")

[HTML]   Dropping 6 rows with invalid doc IDs:
    IL0326R0300EFFDEC2012
    IL0326R0300EFFJULY2012
    OK0256R0404MARCH2015
[HTML] Deduplication: 14776 rows -> 13592 (exact) -> 13455 (by doc+service)
  Dropped 1184 fully duplicate rows
  Dropped 137 duplicate (doc_id, service_name) rows (kept first)
[HTML]   document_id normalized: 22 cells
[HTML]   invalid doc IDs dropped: 6 cells
[HTML]   service_name bad values dropped: 26 cells
[HTML]   renewal leaked header: 92 cells
[HTML]   hcbs_taxonomy_1a typo: 10 cells
[HTML]   provision_of_personal_care capitalisation: 7276 cells
[HTML]   other_state_policies capitalisation: 4400 cells
[HTML] Text cleaning applied to: ['service_name', 'limits_on_the_service', 'provision_of_personal_care_description', 'other_state_policies_description', 'geographic_limitations', 'limited_implementation', 'service_definition', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2a']
[HTML] Final shape: (13429, 29)

[TEXT]   Dropping 2 rows with invalid doc IDs:
    IL0326R030

In [25]:
df_text_clean.query('document_id == "CO0006R0600"')

,document_id,proposed_effective_date,approved_effective_date,service_name,renewal_or_new_or_replacement,limits_on_the_service,service_delivery_method,where_service_provided,provision_of_personal_care,provision_of_personal_care_description,...,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,service_self_directed,service_providermanaged,serviceprovider_lrp,serviceprovider_relative,serviceprovider_lg
789,CO0006R0600,07/01/08,07/01/08,Alternative Care Facility (ACF),Service is included in approved waiver. There ...,Alternative care facilities offered in this wa...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Alternative care facilities (ACF) shall provid...,1,0,0,0,0
790,CO0006R0600,07/01/08,07/01/08,Community Transition Service (CTS),Service is included in approved waiver. There ...,"CTS shall not exceed $2,000.00 per eligible cl...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Services that are provided by a Transition Coo...,0,0,0,0,0
791,CO0006R0600,07/01/08,07/01/08,Consumer Directed Attendant Support Services (...,NaN,Consumer Directed Attendant Support Services o...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Services that assist an individual to accompli...,0,0,0,0,0
792,CO0006R0600,07/01/08,07/01/08,Home Modification,Service is included in approved waiver. There ...,Home modifications are limitedbased on the cli...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"Those physical adaptations tothe home, require...",1,0,0,1,0
793,CO0006R0600,07/01/08,07/01/08,In Home Support Services (IHSS),Service is included in approved waiver. There ...,In Home Support Services offered in this waive...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Services that are provided by an attendant and...,0,1,0,0,0
794,CO0006R0600,07/01/08,07/01/08,Medication Reminder,Service is included in approved waiver. There ...,Itemsreimbursed as Medication Remindershall be...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,(The service title approved in the current wai...,1,0,0,0,0
795,CO0006R0600,07/01/08,07/01/08,Non Medical Transportation,Service is included in approved waiver. There ...,"Effective, , excluding transportation to Adult...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Service offered in order to enable individuals...,1,0,0,0,0
796,CO0006R0600,07/01/08,07/01/08,Personal Emergency Response Systems(PERS),Service is included in approved waiver. There ...,PERS services are limitedto those individuals ...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"PERS is an electronic device, which enables ce...",0,0,0,0,0


In [26]:
# Merge and save
df_merged = merge_service_level(df_html_clean, df_text_clean, verbose=True)
df_merged

HTML docs: 1005, Text docs: 698
  HTML-only: 425
  Text-only: 118
  Overlap:   580

Overlap merge:
  Matched service pairs:  6539
  HTML-only services:     1204
  Text-only services:     2571

Final merged: 17930 rows, 1123 docs
Merge source breakdown:
  merged: 6539
  html_only: 5672
  txt_only_svc: 2571
  txt_only: 1944
  html_only_svc: 1204


,document_id,proposed_effective_date,approved_effective_date,service_name,renewal_or_new_or_replacement,limits_on_the_service,provision_of_personal_care,provision_of_personal_care_description,other_state_policies,other_state_policies_description,...,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,service_self_directed,service_providermanaged,serviceprovider_lrp,serviceprovider_relative,serviceprovider_lg,_merge_source
0,AK0261R0301,7/1/08,7/1/08,Adult Day Services,Service is included in approved waiver. The se...,NaN,No. The State does not make payment to legally...,NaN,The State does not make payment to relatives/l...,NaN,...,NaN,NaN,NaN,Services designed to provide a variety of heal...,0,1,0,0,0,html_only
1,AK0261R0301,7/1/08,7/1/08,Care Coordination,Service is included in approved waiver. The se...,Care Coordination is billed at 1 unit per mont...,No. The State does not make payment to legally...,NaN,The State does not make payment to relatives/l...,NaN,...,NaN,NaN,NaN,Care coordinators assist individuals who apply...,0,1,0,0,0,html_only
2,AK0261R0301,7/1/08,7/1/08,Respite,Service is included in approved waiver. The se...,Under 7 AAC 43.1049: Hourly respite is limited...,No. The State does not make payment to legally...,NaN,The State does not make payment to relatives/l...,NaN,...,NaN,NaN,NaN,Respite care is a service that may be provided...,0,1,0,0,0,html_only
3,AK0261R0301,7/1/08,7/1/08,Chore,Service is included in approved waiver. The se...,5 hours per week unless the recipient has a do...,No. The State does not make payment to legally...,NaN,The State does not make payment to relatives/l...,NaN,...,NaN,NaN,NaN,Chore services includes performing heavy house...,0,1,0,0,0,html_only
4,AK0261R0301,7/1/08,7/1/08,Environmental Modifications,Service is included in approved waiver. The se...,Environmental modifications may not exceed $10...,No. The State does not make payment to legally...,NaN,The State does not make payment to relatives/l...,NaN,...,NaN,NaN,NaN,Environmental modification services are physic...,0,1,0,0,0,html_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17925,WY0370R0302,1/1/18,04/01/18,Adult Day Services,NaN,Adult Day Services are not a habilitation serv...,No. The State does not make payment to legally...,NaN,Relatives/legal guardians may be paid for prov...,Limitations on the types of relatives who may ...,...,04050 adult day health,NaN,NaN,Adult Day Services are not a habilitation serv...,0,1,0,1,0,merged
17926,WY0370R0302,1/1/18,04/01/18,"Speech, Hearing and Language Services",NaN,A minimum of 45 minutes of service per session...,No. The State does not make payment to legally...,NaN,Relatives/legal guardians may be paid for prov...,Limitations on the types of relatives who may ...,...,"11100 speech, hearing, and language therapy",NaN,NaN,"Speech, Hearing and Language services consist ...",0,1,0,0,0,merged
17927,WY0370R0302,1/1/18,04/01/18,Case Management,NaN,Case Managers shall be reimbursed up to 1 unit...,No. The State does not make payment to legally...,NaN,Relatives/legal guardians may be paid for prov...,Limitations on the types of relatives who may ...,...,01010 case management,NaN,NaN,Case management is a service to assist partici...,0,1,0,0,0,merged
17928,WY0370R0302,1/1/18,04/01/18,Personal Care,NaN,This service is available to all ages and is a...,No. The State does not make payment to legally...,NaN,Relatives/legal guardians may be paid for prov...,Limitations on the types of relatives who may ...,...,08030 personal care,NaN,NaN,A range of assistance to enable waiver partici...,1,1,0,1,0,merged


In [27]:
df_merged.query('document_id == "CO0006R0600"')

,document_id,proposed_effective_date,approved_effective_date,service_name,renewal_or_new_or_replacement,limits_on_the_service,provision_of_personal_care,provision_of_personal_care_description,other_state_policies,other_state_policies_description,...,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,service_self_directed,service_providermanaged,serviceprovider_lrp,serviceprovider_relative,serviceprovider_lg,_merge_source
7800,CO0006R0600,7/1/08,07/01/08,Non Medical Transportation,Service is included in approved waiver. There ...,"Effective, , excluding transportation to Adult...",Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,Service offered in order to enable individuals...,1,1,0,0,0,merged
7801,CO0006R0600,7/1/08,07/01/08,In Home Support Services (IHSS),Service is included in approved waiver. There ...,In Home Support Services offered in this waive...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,Services that are provided by an attendant and...,1,1,0,1,1,merged
7802,CO0006R0600,7/1/08,07/01/08,Personal Emergency Response Systems (PERS),Service is included in approved waiver. There ...,PERS services are limitedto those individuals ...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,"PERS is an electronic device, which enables ce...",0,1,0,0,0,merged
7803,CO0006R0600,7/1/08,07/01/08,Alternative Care Facility (ACF),Service is included in approved waiver. There ...,Alternative care facilities offered in this wa...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,Alternative care facilities (ACF) shall provid...,1,1,0,0,0,merged
7804,CO0006R0600,7/1/08,07/01/08,Community Transition Service (CTS),Service is included in approved waiver. There ...,"CTS shall not exceed $2,000.00 per eligible cl...",Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,Services that are provided by a Transition Coo...,0,1,0,0,0,merged
7805,CO0006R0600,7/1/08,07/01/08,Consumer Directed Attendant Support Services (...,NaN,Consumer Directed Attendant Support Services o...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,Services that assist an individual to accompli...,0,0,1,1,1,merged
7806,CO0006R0600,7/1/08,07/01/08,Home Modification,Service is included in approved waiver. There ...,Home modifications are limitedbased on the cli...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,"Those physical adaptations tothe home, require...",1,1,1,1,1,merged
7807,CO0006R0600,7/1/08,07/01/08,Medication Reminder,Service is included in approved waiver. There ...,Itemsreimbursed as Medication Remindershall be...,Yes. The State makes payment to legally respon...,A spouse may be paid to furnish extraordinary ...,The State makes payment to relatives/legal gua...,For the purpose of this section family shall b...,...,NaN,NaN,NaN,(The service title approved in the current wai...,1,1,0,0,0,merged
7808,CO000

In [ ]:
OUTPUT = "../output/merged_service_level.csv"
df_merged.to_csv(OUTPUT, index=False, quoting=csv.QUOTE_ALL)
print(f"\nSaved {df_merged.shape[0]:,} rows x {df_merged.shape[1]} cols -> {OUTPUT}")


Saved 17,930 rows x 30 cols -> ../output/merged_service_level.csv


In [30]:
df_merged.columns

Index(['document_id', 'proposed_effective_date', 'approved_effective_date',
       'service_name', 'renewal_or_new_or_replacement',
       'limits_on_the_service', 'provision_of_personal_care',
       'provision_of_personal_care_description', 'other_state_policies',
       'other_state_policies_description', 'waive_statewideness',
       'geographic_limitations', 'limited_implementation',
       'year_1_participants', 'year_2_participants', 'year_3_participants',
       'year_4_participants', 'year_5_participants', 'service_type',
       'hcbs_taxonomy_1', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2',
       'hcbs_taxonomy_2a', 'service_definition', 'service_self_directed',
       'service_providermanaged', 'serviceprovider_lrp',
       'serviceprovider_relative', 'serviceprovider_lg', '_merge_source'],
      dtype='object')

In [31]:
final_col = ['document_id', 'proposed_effective_date', 'approved_effective_date',
       'service_name','service_type', 'renewal_or_new_or_replacement',
       'limits_on_the_service', 'service_self_directed',
       'service_providermanaged', 'serviceprovider_lrp',
       'serviceprovider_relative', 'serviceprovider_lg','provision_of_personal_care',
       'provision_of_personal_care_description', 'other_state_policies',
       'other_state_policies_description', 'waive_statewideness',
       'geographic_limitations', 'limited_implementation',
       'year_1_participants', 'year_2_participants', 'year_3_participants',
       'year_4_participants', 'year_5_participants', 
       'hcbs_taxonomy_1', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2',
       'hcbs_taxonomy_2a', 'service_definition']
df_merged_clean = df_merged[final_col]
df_merged_clean

,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,year_1_participants,year_2_participants,year_3_participants,year_4_participants,year_5_participants,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition
0,AK0261R0301,7/1/08,7/1/08,Adult Day Services,Statutory Service,Service is included in approved waiver. The se...,NaN,0,1,0,...,2480,2580,2680,2780,2890,NaN,NaN,NaN,NaN,Services designed to provide a variety of heal...
1,AK0261R0301,7/1/08,7/1/08,Care Coordination,Statutory Service,Service is included in approved waiver. The se...,Care Coordination is billed at 1 unit per mont...,0,1,0,...,2480,2580,2680,2780,2890,NaN,NaN,NaN,NaN,Care coordinators assist individuals who apply...
2,AK0261R0301,7/1/08,7/1/08,Respite,Statutory Service,Service is included in approved waiver. The se...,Under 7 AAC 43.1049: Hourly respite is limited...,0,1,0,...,2480,2580,2680,2780,2890,NaN,NaN,NaN,NaN,Respite care is a service that may be provided...
3,AK0261R0301,7/1/08,7/1/08,Chore,Other Service,Service is included in approved waiver. The se...,5 hours per week unless the recipient has a do...,0,1,0,...,2480,2580,2680,2780,2890,NaN,NaN,NaN,NaN,Chore services includes performing heavy house...
4,AK0261R0301,7/1/08,7/1/08,Environmental Modifications,Other Service,Service is included in approved waiver. The se...,Environmental modifications may not exceed $10...,0,1,0,...,2480,2580,2680,2780,2890,NaN,NaN,NaN,NaN,Environmental modification services are physic...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17925,WY0370R0302,1/1/18,04/01/18,Adult Day Services,Statutory Service,NaN,Adult Day Services are not a habilitation serv...,0,1,0,...,240,240,240,240,1,04 Day Services,04050 adult day health,NaN,NaN,Adult Day Services are not a habilitation serv...
17926,WY0370R0302,1/1/18,04/01/18,"Speech, Hearing and Language Services",Extended State Plan Service,NaN,A minimum of 45 minutes of service per session...,0,1,0,...,240,240,240,240,1,11 Other Health and Therapeutic Services,"11100 speech, hearing, and language therapy",NaN,NaN,"Speech, Hearing and Language services consist ..."
17927,WY0370R0302,1/1/18,04/01/18,Case Management,Statutory Service,NaN,Case Managers shall be reimbursed up to 1 unit...,0,1,0,...,240,240,240,240,1,01 Case Management,01010 case management,NaN,NaN,Case management is a service to assist partici...
17928,WY0370R0302,1/1/18,04/01/18,Personal Care,Statutory Service,NaN,This service is available to all ages and is a...,1,1,0,...,240,240,240,240,1,08 Home-Based Services,08030 personal care,NaN,NaN,A range of assistance to enable waiver partici...


In [33]:
df_merged_clean['provision_of_personal_care'].value_counts(dropna=False)

provision_of_personal_care
No. The State does not make payment to legally responsible individuals for furnishing personal care or similar services.                                             12115
Yes. The State makes payment to legally responsible individuals for furnishing personal care or similar services when they are qualified to provide the services.     5685
NaN                                                                                                                                                                    130
Name: count, dtype: int64

In [32]:
df_corpus = pd.read_csv("../output/service_level_corpus.csv", dtype=str, low_memory=False)
df_corpus

,document_id,proposed_effective_date,approved_effective_date,service_name,renewal_or_new_or_replacement,limits_on_the_service,service_delivery_method,where_service_provided,provision_of_personal_care,provision_of_personal_care_description,...,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,service_self_directed,service_providermanaged,serviceprovider_lrp,serviceprovider_relative,serviceprovider_lg
0,VA0358R0504,03/01/24,03/01/24,Group Day Services,NaN,The service unit is an hour. Group day service...,['Provider managed'],NaN,1,Legally responsible individuals may provide th...,...,04 Day Services,04020 day habilitation,NaN,NaN,(Scope):\nGroup Day Services include skill bui...,0,1,0,0,0
1,VA0358R0504,03/01/24,03/01/24,In-Home Support Services,NaN,The unit of service is an hour. The hours to b...,['Provider managed'],"['Relative', 'Legal Guardian']",1,Legally responsible individuals may provide th...,...,08 Home-Based Services,08010 home-based habilitation,NaN,NaN,(Scope):\nIn-Home Support services are residen...,0,1,0,1,1
2,VA0358R0504,03/01/24,03/01/24,Individual Supported Employment,NaN,The service unit for individual supported empl...,['Provider managed'],NaN,1,Legally responsible individuals may provide th...,...,03 Supported Employment,03010 job development,03 Supported Employment,"03021 ongoing supported employment, individual",(Scope):\nIndividual Supported Employment serv...,0,1,0,0,0
3,VA0358R0504,03/01/24,03/01/24,Personal Assistance Services,NaN,The unit of service is an hour. The hours to b...,"['Participant-directed', 'Provider managed']","['Legally Responsible Person', 'Relative', 'Le...",1,Legally responsible individuals may provide th...,...,08 Home-Based Services,08030 personal care,NaN,NaN,(Scope):\nPersonal assistance services mean di...,1,1,1,1,1
4,VA0358R0504,03/01/24,03/01/24,Respite Services,NaN,Respite is limited to 480 hours per individual...,"['Participant-directed', 'Provider managed']","['Relative', 'Legal Guardian']",1,Legally responsible individuals may provide th...,...,09 Caregiver Support,"09011 respite, out-of-home",09 Caregiver Support,"09012 respite, in-home",(Scope):\nRespite services are specifically de...,1,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13555,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Exploration,NaN,After an individual has received the Explorati...,['Provider managed'],['Legally Responsible Person'],1,Family members may be employed by a provider a...,...,NaN,NaN,NaN,NaN,(Scope):\nSupported Employment Individual-Expl...,0,1,1,0,0
13556,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Job Coaching,NaN,JC services do not include supports for volunt...,['Provider managed'],['Legally Responsible Person'],1,Family members may be employed by a provider a...,...,NaN,NaN,NaN,NaN,(Scope):\nSupported Employment Individual-Job ...,0,1,1,0,0
13557,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Job Development,NaN,o JD Plan or SE Plan: After an individual has ...,['Provider managed'],['Legally Responsible Person'],1,Family members may be employed by a provider a...,...,NaN,NaN,NaN,NaN,(Scope):\nSupported Employment Individual-Job ...,0,1,1,0,0
13558,TN0128R0608,07/01/24,07/01/24,Supported Living,NaN,A person authorized to receive the RSNA-HB pay...,['Provider managed'],NaN,1,Family members may be employed by a provider a...,...,NaN,NaN,NaN,NaN,(Scope):\nSupported Living (SL) shall mean a t...,0,1,0,0,0


In [34]:
import re

_PPC_NO  = "No. The State does not make payment to legally responsible individuals for furnishing personal care or similar services."
_PPC_YES = "Yes. The State makes payment to legally responsible individuals for furnishing personal care or similar services when they are qualified to provide the services."

def prep_corpus(df):
    df = df.copy()

    # rename is_statewide -> waive_statewideness, 0/1 -> No/Yes
    if "is_statewide" in df.columns:
        df = df.rename(columns={"is_statewide": "waive_statewideness"})
        df["waive_statewideness"] = df["waive_statewideness"].map(
            {"0": "No", "1": "Yes", 0: "No", 1: "Yes"}
        ).fillna(df["waive_statewideness"])

    # provision_of_personal_care 0/1 -> full sentence
    if "provision_of_personal_care" in df.columns:
        df["provision_of_personal_care"] = df["provision_of_personal_care"].map(
            {"0": _PPC_NO, "1": _PPC_YES, 0: _PPC_NO, 1: _PPC_YES}
        ).fillna(df["provision_of_personal_care"])

    # strip leading "(Scope):" from service_definition
    if "service_definition" in df.columns:
        df["service_definition"] = (
            df["service_definition"]
            .str.replace(r"^\s*\(Scope\)\s*:?\s*", "", regex=True)
            .str.strip()
        )

    # run standard cleaning (encoding, boilerplate, dedup, invalid IDs)
    df = clean_service_level_dataframe(df, source="text", verbose=True)

    # keep only final_col columns that exist
    keep = [c for c in final_col if c in df.columns]
    missing = [c for c in final_col if c not in df.columns]
    if missing:
        print(f"Corpus missing columns (will be NaN when merged): {missing}")
    return df[keep]
df_corpus_clean = prep_corpus(df_corpus)
df_corpus_clean

[TEXT] Deduplication: 13560 rows -> 13559 (exact) -> 13559 (by doc+service)
  Dropped 1 fully duplicate rows
[TEXT]   hcbs_taxonomy_1a typo: 16 cells
[TEXT]   hcbs_taxonomy_2a typo: 4 cells
[TEXT]   other_state_policies capitalisation: 7280 cells
[TEXT] Text cleaning applied to: ['service_name', 'limits_on_the_service', 'provision_of_personal_care_description', 'other_state_policies_description', 'geographic_limitations', 'limited_implementation', 'service_definition', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2a']
[TEXT] Final shape: (13559, 33)


,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,year_1_participants,year_2_participants,year_3_participants,year_4_participants,year_5_participants,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition
0,VA0358R0504,03/01/24,03/01/24,Group Day Services,Statutory Service,NaN,The service unit is an hour. Group day service...,0,1,0,...,5463,5463,5463,5463,5463,04 Day Services,04020 day habilitation,NaN,NaN,Group Day Services include skill building or s...
1,VA0358R0504,03/01/24,03/01/24,In-Home Support Services,Statutory Service,NaN,The unit of service is an hour. The hours to b...,0,1,0,...,5463,5463,5463,5463,5463,08 Home-Based Services,08010 home-based habilitation,NaN,NaN,In-Home Support services are residential servi...
2,VA0358R0504,03/01/24,03/01/24,Individual Supported Employment,Statutory Service,NaN,The service unit for individual supported empl...,0,1,0,...,5463,5463,5463,5463,5463,03 Supported Employment,03010 job development,03 Supported Employment,"03021 ongoing supported employment, individual",Individual Supported Employment services consi...
3,VA0358R0504,03/01/24,03/01/24,Personal Assistance Services,Statutory Service,NaN,The unit of service is an hour. The hours to b...,1,1,1,...,5463,5463,5463,5463,5463,08 Home-Based Services,08030 personal care,NaN,NaN,Personal assistance services mean direct suppo...
4,VA0358R0504,03/01/24,03/01/24,Respite Services,Statutory Service,NaN,Respite is limited to 480 hours per individual...,1,1,0,...,5463,5463,5463,5463,5463,09 Caregiver Support,"09011 respite, out-of-home",09 Caregiver Support,"09012 respite, in-home",Respite services are specifically designed to ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13554,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Exploration,Other Service,NaN,After an individual has received the Explorati...,0,1,1,...,4648,4508,4373,4308,4243,NaN,NaN,NaN,NaN,Supported Employment Individual-Exploration (S...
13555,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Job Coaching,Other Service,NaN,JC services do not include supports for volunt...,0,1,1,...,4648,4508,4373,4308,4243,NaN,NaN,NaN,NaN,Supported Employment Individual-Job Coaching (...
13556,TN0128R0608,07/01/24,07/01/24,Supported Employment Individual-Job Development,Other Service,NaN,o JD Plan or SE Plan: After an individual has ...,0,1,1,...,4648,4508,4373,4308,4243,NaN,NaN,NaN,NaN,Supported Employment Individual-Job Developmen...
13557,TN0128R0608,07/01/24,07/01/24,Supported Living,Other Service,NaN,A person authorized to receive the RSNA-HB pay...,0,1,0,...,4648,4508,4373,4308,4243,NaN,NaN,NaN,NaN,Supported Living (SL) shall mean a type of res...


In [35]:
import re

def _svc_key(name):
    s = str(name).strip().lower()
    s = re.sub(r"\s*\([^)]*\)", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return s.strip()

# baseline is the html+text merged, restricted to final_col
base = df_merged_clean.copy()

# corpus lookup by (doc_id, service_name_key)
corpus_lookup = {}
for idx, row in df_corpus_clean.iterrows():
    k = (row["document_id"], _svc_key(str(row.get("service_name", ""))))
    corpus_lookup[k] = idx

shared_cols = [c for c in df_corpus_clean.columns if c in base.columns
               and c not in ("document_id", "service_name")]

# 1. Fill empty cells in base where corpus has a value
filled = 0
for idx, row in base.iterrows():
    k = (row["document_id"], _svc_key(str(row.get("service_name", ""))))
    corp_idx = corpus_lookup.get(k)
    if corp_idx is None:
        continue
    corp_row = df_corpus_clean.loc[corp_idx]
    for col in shared_cols:
        if pd.isna(row[col]) or str(row[col]).strip() == "":
            val = corp_row.get(col)
            if pd.notna(val) and str(val).strip() != "":
                base.at[idx, col] = val
                filled += 1

print(f"Cells filled from corpus: {filled:,}")

# 2. Append corpus-only rows (doc_id+service not in base at all)
base_keys = set(
    (row["document_id"], _svc_key(str(row.get("service_name", ""))))
    for _, row in base.iterrows()
)

corpus_only = df_corpus_clean[
    df_corpus_clean.apply(
        lambda r: (r["document_id"], _svc_key(str(r.get("service_name", "")))) not in base_keys,
        axis=1,
    )
].copy()

print(f"Corpus-only rows to append: {len(corpus_only):,}")

df_final = pd.concat([base, corpus_only], ignore_index=True)
df_final = df_final.sort_values(["document_id", "service_name"]).reset_index(drop=True)
print(f"Final shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} cols")


Cells filled from corpus: 33,840
Corpus-only rows to append: 1,300
Final shape: 19,230 rows x 29 cols


In [36]:
df_final

,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,year_1_participants,year_2_participants,year_3_participants,year_4_participants,year_5_participants,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition
0,AK0260R0600,7/1/21,07/01/21,Care Coordination,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,2120,2120,2130,2140,2150,01 Case Management,01010 case management,NaN,NaN,Care Coordinators assist individuals to gain a...
1,AK0260R0600,7/1/21,07/01/21,Day Habilitation,Statutory Service,Service is included in approved waiver. There ...,"Per 7 AAC 130.260, the department will only pa...",0,1,0,...,2120,2120,2130,2140,2150,04 Day Services,04020 day habilitation,NaN,NaN,Day habilitation services may be provided to a...
2,AK0260R0600,7/1/21,07/01/21,Employment Services,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,2120,2120,2130,2140,2150,03 Supported Employment,"03021 ongoing supported employment, individual",03 Supported Employment,"03022 ongoing supported employment, group",Employment services assist recipients to acqui...
3,AK0260R0600,7/1/21,07/01/21,Environmental Modifications,Other Service,Service is included in approved waiver. There ...,The State will pay for environmental modificat...,0,1,0,...,2120,2120,2130,2140,2150,NaN,NaN,NaN,NaN,Environmental modification services may be pro...
4,AK0260R0600,7/1/21,07/01/21,Intensive Active Treatment,Other Service,Service is included in approved waiver. There ...,NaN,0,1,0,...,2120,2120,2130,2140,2150,NaN,NaN,NaN,NaN,Intensive Active Treatment (IAT) assists parti...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19225,WY1061R0200,04/01/24,04/01/24,Special Family Habilitation Home,Other Service,Service is included in approved waiver. There ...,This service is not open to new participants w...,0,1,0,...,1835,1835,1835,1835,1835,02 Round-the-Clock Services,02031 in-home residential habilitation,NaN,NaN,Special family habilitation home (SFHH) servic...
19226,WY1061R0200,04/01/24,04/01/24,Specialized Equipment,Other Service,Service is included in approved waiver. There ...,"Specialized Equipment has a $4,000 annual limi...",0,1,0,...,1835,1835,1835,1835,1835,"14 Equipment, Technology, and Modifications",14031 equipment and technology,NaN,NaN,"Specialized Equipment includes: - Devices, con..."
19227,WY1061R0200,04/01/24,04/01/24,"Speech, Hearing and Language Services",Extended State Plan Service,Service is included in approved waiver. There ...,"Speech, Hearing, and Language Services are ava...",0,1,0,...,1835,1835,1835,1835,1835,11 Other Health and Therapeutic Services,"11100 speech, hearing, and language therapy",NaN,NaN,"Speech, Hearing, and Language Services consist..."
19228,WY1061R0200,04/01/24,04/01/24,Supported Employment,Statutory Service,Service is included in approved waiver. There ...,Documentation that the service is not availabl...,1,1,0,...,1835,1835,1835,1835,1835,03 Supported Employment,03010 job development,03 Supported Employment,"03021 ongoing supported employment, individual",Supported Employment Services are intended to ...


In [37]:
df_final.query('document_id == "CO0006R0600"')

,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,year_1_participants,year_2_participants,year_3_participants,year_4_participants,year_5_participants,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition
1204,CO0006R0600,7/1/08,7/1/08,Adult Day Health,Statutory Service,Service is included in approved waiver. There ...,Adult Day Health services offered in this waiv...,0,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Services furnished 4 or more hours per day on ...
1205,CO0006R0600,7/1/08,07/01/08,Alternative Care Facility (ACF),Other Service,Service is included in approved waiver. There ...,Alternative care facilities offered in this wa...,1,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Alternative care facilities (ACF) shall provid...
1206,CO0006R0600,7/1/08,07/01/08,Community Transition Service (CTS),Other Service,Service is included in approved waiver. There ...,"CTS shall not exceed $2,000.00 per eligible cl...",0,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Services that are provided by a Transition Coo...
1207,CO0006R0600,7/1/08,07/01/08,Consumer Directed Attendant Support Services (...,Other Service,Service is included in approved waiver. There ...,Consumer Directed Attendant Support Services o...,0,0,1,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Services that assist an individual to accompli...
1208,CO0006R0600,7/1/08,07/01/08,Home Modification,Other Service,Service is included in approved waiver. There ...,Home modifications are limitedbased on the cli...,1,1,1,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,"Those physical adaptations tothe home, require..."
1209,CO0006R0600,7/1/08,7/1/08,Homemaker,Statutory Service,Service is included in approved waiver. There ...,Family members shall not be reimbursed to prov...,0,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Services consisting of general household activ...
1210,CO0006R0600,7/1/08,07/01/08,In Home Support Services (IHSS),Other Service,Service is included in approved waiver. There ...,In Home Support Services offered in this waive...,1,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Services that are provided by an attendant and...
1211,CO0006R0600,7/1/08,07/01/08,Medication Reminder,Other Service,Service is included in approved waiver. There ...,Itemsreimbursed as Medication Remindershall be...,1,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,(The service title approved in the current wai...
1212,CO0006R0600,7/1/08,07/01/08,Non Medical Transportation,Other Service,Service is included in approved waiver. There ...,"Effective, , excluding transportation to Adult...",1,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,Service offered in order to enable individuals...
1213,CO0006R0600,7/1/08,7/1/08,Personal Care,Statutory Service,Service is included in approved waiver. There ...,Personal Care providers may be members of the ...,0,1,0,...,22384,22384,22384,22384,22384,NaN,NaN,NaN,NaN,"Assistance with eating, bathing, dressing, per..."


In [38]:
print(f"\nFinal merged shape: {df_final.shape[0]:,} rows x {df_final.shape[1]} cols")

OUTPUT_FINAL = "../output/merged_with_corpus_service_level.csv"
df_final.to_csv(OUTPUT_FINAL, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved -> {OUTPUT_FINAL}")


Final merged shape: 19,230 rows x 29 cols
Saved -> ../output/merged_with_corpus_service_level.csv


#### Categorization of service types

In [39]:
import sys, csv
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import importlib
import categorizer.hcbs_service_categorizer as _hcbs_mod
import categorizer.limit_categorizer as _limit_mod
importlib.reload(_hcbs_mod)
importlib.reload(_limit_mod)

from categorizer.hcbs_service_categorizer import HCBSServiceCategorizer
from categorizer.limit_categorizer import categorize_dataframe, print_summary

# ── 1. HCBS service taxonomy categorization (based on service_name) ──────────
cat = HCBSServiceCategorizer(output_dir="../output/categorizer_outputs")
df_cat = cat.process_data(df_final)

# columns added: taxonomy_code, taxonomy_category, taxonomy_service,
#                confidence, match_type, categorization_source, needs_manual_review

# ── 2. Limit categorization (based on limits_on_the_service) ─────────────────
df_cat = categorize_dataframe(df_cat, limit_col="limits_on_the_service")

# columns added: limit_categories, limit_type_primary

print_summary(df_cat)


2026-06-19 19:08:52,085 - categorizer.hcbs_service_categorizer - INFO - Initializing HCBS Taxonomy Classifier with Enhanced Matching
2026-06-19 19:08:52,140 - categorizer.hcbs_service_categorizer - INFO - Output directory: ../output/categorizer_outputs
2026-06-19 19:08:52,141 - categorizer.hcbs_service_categorizer - INFO - Starting service categorization with enhanced matching
2026-06-19 19:08:52,658 - categorizer.hcbs_service_categorizer - INFO - Processed 100/19230 services
2026-06-19 19:08:52,674 - categorizer.hcbs_service_categorizer - INFO - Processed 200/19230 services
2026-06-19 19:08:52,689 - categorizer.hcbs_service_categorizer - INFO - Processed 300/19230 services
2026-06-19 19:08:52,706 - categorizer.hcbs_service_categorizer - INFO - Processed 400/19230 services
2026-06-19 19:08:52,732 - categorizer.hcbs_service_categorizer - INFO - Processed 500/19230 services
2026-06-19 19:08:52,760 - categorizer.hcbs_service_categorizer - INFO - Processed 600/19230 services
2026-06-19 19:

Service Limit Categorization Summary
Total rows: 19230
Categorized: 17965 (93.4%)
Empty/missing: 1265

Primary category distribution:
  Cost / Budget Limits                       6671  (34.7%)
  Hours / Time Restrictions                  3218  (16.7%)
  Other / Unclassified                       3062  (15.9%)
  Location / Setting Restrictions            2259  (11.7%)
  Medical / Clinical Criteria                 775  (4.0%)
  Frequency / Quantity Caps                   473  (2.5%)
  Authorization Required                      393  (2.0%)
  Documentation Required                      337  (1.8%)
  Prohibition / Exclusion Rules               189  (1.0%)
  Participant / Eligibility Rules             165  (0.9%)
  Extraction Artifact                         157  (0.8%)
  Provider Managed                            148  (0.8%)
  No Explicit Cap / None                      118  (0.6%)


In [40]:
df_cat

,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,needs_manual_review,categorization_source,taxonomy_code,taxonomy_category,taxonomy_service,confidence,match_type,warning,limit_categories,limit_type_primary
0,AK0260R0600,7/1/21,07/01/21,Care Coordination,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,False,service_name,1,Case Management,Case Management,high,exact_name,NaN,,
1,AK0260R0600,7/1/21,07/01/21,Day Habilitation,Statutory Service,Service is included in approved waiver. There ...,"Per 7 AAC 130.260, the department will only pa...",0,1,0,...,False,service_name,4,Day Services,Day Habilitation,high,exact_name,NaN,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
2,AK0260R0600,7/1/21,07/01/21,Employment Services,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,False,service_name_keyword,3,Supported Employment,Supported Employment - Unspecified,medium_high,keyword_name,NaN,,
3,AK0260R0600,7/1/21,07/01/21,Environmental Modifications,Other Service,Service is included in approved waiver. There ...,The State will pay for environmental modificat...,0,1,0,...,False,service_name,14,"Equipment, Technology, and Modifications",Home/Vehicle Accessibility Adaptations,high,exact_name,NaN,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
4,AK0260R0600,7/1/21,07/01/21,Intensive Active Treatment,Other Service,Service is included in approved waiver. There ...,NaN,0,1,0,...,False,service_name,4,Day Services,Community Integration,high,exact_name,NaN,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19225,WY1061R0200,04/01/24,04/01/24,Special Family Habilitation Home,Other Service,Service is included in approved waiver. There ...,This service is not open to new participants w...,0,1,0,...,False,revised_manual_mapping,2,Round-the-Clock Services,Special Family Habilitation Home,medium_high,revised_mapping,NaN,"Location / Setting Restrictions , Prohibition ...",Location / Setting Restrictions
19226,WY1061R0200,04/01/24,04/01/24,Specialized Equipment,Other Service,Service is included in approved waiver. There ...,"Specialized Equipment has a $4,000 annual limi...",0,1,0,...,False,service_name,14,"Equipment, Technology, and Modifications",Equipment and Technology,high,exact_name,NaN,"Hours / Time Restrictions , Frequency / Quanti...",Cost / Budget Limits
19227,WY1061R0200,04/01/24,04/01/24,"Speech, Hearing and Language Services",Extended State Plan Service,Service is included in approved waiver. There ...,"Speech, Hearing, and Language Services are ava...",0,1,0,...,False,revised_manual_mapping,11,Other Health and Therapeutic Services,"Speech, Hearing and Language Services",medium_high,revised_mapping,NaN,"Frequency / Quantity Caps , Documentation Requ...",Extraction Artifact
19228,WY1061R0200,04/01/24,04/01/24,Supported Employment,Statutory Service,Service is included in approved waiver. There ...,Documentation that the service is not availabl...,1,1,0,...,False,service_name_keyword,3,Supported Employment,Ongoing Supported Employment - Individual,medium_high,keyword_name,NaN,"Documentation Required , Prohibition / Exclusi...",Documentation Required


In [41]:
final_col_v2 = final_col + [
    'taxonomy_code',
    'taxonomy_category',
    'taxonomy_service',
    'limit_categories',
    'limit_type_primary',
]

keep = [c for c in final_col_v2 if c in df_cat.columns]
df_final_clean = df_cat[keep]
print(f"Shape: {df_final_clean.shape[0]:,} rows x {df_final_clean.shape[1]} cols")
df_final_clean


Shape: 19,230 rows x 34 cols


,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,taxonomy_code,taxonomy_category,taxonomy_service,limit_categories,limit_type_primary
0,AK0260R0600,7/1/21,07/01/21,Care Coordination,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,01 Case Management,01010 case management,NaN,NaN,Care Coordinators assist individuals to gain a...,1,Case Management,Case Management,,
1,AK0260R0600,7/1/21,07/01/21,Day Habilitation,Statutory Service,Service is included in approved waiver. There ...,"Per 7 AAC 130.260, the department will only pa...",0,1,0,...,04 Day Services,04020 day habilitation,NaN,NaN,Day habilitation services may be provided to a...,4,Day Services,Day Habilitation,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
2,AK0260R0600,7/1/21,07/01/21,Employment Services,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,03 Supported Employment,"03021 ongoing supported employment, individual",03 Supported Employment,"03022 ongoing supported employment, group",Employment services assist recipients to acqui...,3,Supported Employment,Supported Employment - Unspecified,,
3,AK0260R0600,7/1/21,07/01/21,Environmental Modifications,Other Service,Service is included in approved waiver. There ...,The State will pay for environmental modificat...,0,1,0,...,NaN,NaN,NaN,NaN,Environmental modification services may be pro...,14,"Equipment, Technology, and Modifications",Home/Vehicle Accessibility Adaptations,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
4,AK0260R0600,7/1/21,07/01/21,Intensive Active Treatment,Other Service,Service is included in approved waiver. There ...,NaN,0,1,0,...,NaN,NaN,NaN,NaN,Intensive Active Treatment (IAT) assists parti...,4,Day Services,Community Integration,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19225,WY1061R0200,04/01/24,04/01/24,Special Family Habilitation Home,Other Service,Service is included in approved waiver. There ...,This service is not open to new participants w...,0,1,0,...,02 Round-the-Clock Services,02031 in-home residential habilitation,NaN,NaN,Special family habilitation home (SFHH) servic...,2,Round-the-Clock Services,Special Family Habilitation Home,"Location / Setting Restrictions , Prohibition ...",Location / Setting Restrictions
19226,WY1061R0200,04/01/24,04/01/24,Specialized Equipment,Other Service,Service is included in approved waiver. There ...,"Specialized Equipment has a $4,000 annual limi...",0,1,0,...,"14 Equipment, Technology, and Modifications",14031 equipment and technology,NaN,NaN,"Specialized Equipment includes: - Devices, con...",14,"Equipment, Technology, and Modifications",Equipment and Technology,"Hours / Time Restrictions , Frequency / Quanti...",Cost / Budget Limits
19227,WY1061R0200,04/01/24,04/01/24,"Speech, Hearing and Language Services",Extended State Plan Service,Service is included in approved waiver. There ...,"Speech, Hearing, and Language Services are ava...",0,1,0,...,11 Other Health and Therapeutic Services,"11100 speech, hearing, and language therapy",NaN,NaN,"Speech, Hearing, and Language Services consist...",11,Other Health and Therapeutic Services,"Speech, Hearing and Language Services","Frequency / Quantity Caps , Documentation Requ...",Extraction Artifact
19228,WY1061R0200,04/01/24,04/01/24,Supported Employment,Statutory Service,Service is included in approved waiver. There ...,Documentation that the service is not availabl...,1,1,0,...,03 Supported Employment,03010 job development,03 Supported Employment,"03021 ongoing supported employment, individual",Supported Employment Services are intended to ...,3,Supported Employment,Ongoing Supported Employment - Individua

In [43]:
# missing service_name count
missing_svc = df_final_clean['service_name'].isna() | (df_final_clean['service_name'].str.strip() == '')
print(f"Missing service_name: {missing_svc.sum():,} / {len(df_final_clean):,} rows")

# sample of rows with high missingness (>= 80% of non-key cols empty)
key_cols = {'document_id', 'service_name'}
data_cols = [c for c in df_final_clean.columns if c not in key_cols]

missing_pct = df_final_clean[data_cols].apply(
    lambda row: row.apply(lambda v: pd.isna(v) or str(v).strip() == '').sum() / len(data_cols) * 100,
    axis=1
)

threshold = 90  # change to 90 if you want stricter
sparse_rows = df_final_clean[missing_pct >= threshold].copy()
sparse_rows['_missing_pct'] = missing_pct[missing_pct >= threshold].round(1)

print(f"Rows with >={threshold}% missing data: {len(sparse_rows):,}")
sparse_rows.sort_values('_missing_pct', ascending=False)


Missing service_name: 3 / 19,230 rows
Rows with >=90% missing data: 0


,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,taxonomy_code,taxonomy_category,taxonomy_service,limit_categories,limit_type_primary,_missing_pct


In [64]:
# final layer cleaning

import re

def final_layer_clean(df):
    df = df.copy()

    # 1. geographic_limitations and limited_implementation: 0/1 → empty
    for col in ('geographic_limitations', 'limited_implementation'):
        if col in df.columns:
            df[col] = df[col].apply(
                lambda v: '' if str(v).strip() in ('0', '1', '0.0', '1.0') else v
            )

    # 2. Any text column: ". ." or ".  " or lone "." patterns → empty
    dot_re = re.compile(r'^\s*(\.\s*)+\s*$')
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(
                lambda v: '' if pd.notna(v) and dot_re.match(str(v)) else v
            )

    # 3. limited_implementation: garbage values → empty
    # matches "geographic area: ..." or "by geographic area: ..." with no real content after colon
    geo_junk_re = re.compile(r'^\s*(by\s+)?geographic\s+area\s*:\s*[\s.\-]*$', re.IGNORECASE)
    assurances_re = re.compile(r'^\s*assurances\s*$', re.IGNORECASE)
    if 'limited_implementation' in df.columns:
        df['limited_implementation'] = df['limited_implementation'].apply(
            lambda v: '' if pd.notna(v) and (geo_junk_re.match(str(v)) or assurances_re.match(str(v))) else v
        )

    # 4. hcbs taxonomy cols: remove dates and dot-only values
    date_re = re.compile(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b')
    tax_cols = [c for c in ('hcbs_taxonomy_1', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2', 'hcbs_taxonomy_2a') if c in df.columns]
    for col in tax_cols:
        df[col] = df[col].apply(lambda v: '' if pd.notna(v) and date_re.search(str(v)) else v)
        df[col] = df[col].apply(lambda v: '' if pd.notna(v) and dot_re.match(str(v)) else v)

    # 5. service_name: "Service Title:" or "Provider Type:" → empty
    svc_junk_re = re.compile(r'^\s*(service\s+title|provider\s+type)\s*:.*$', re.IGNORECASE)
    if 'service_name' in df.columns:
        df['service_name'] = df['service_name'].apply(
            lambda v: '' if pd.notna(v) and svc_junk_re.match(str(v)) else v
        )

    # 6. limits_on_the_service: standalone ":" → empty
    if 'limits_on_the_service' in df.columns:
        df['limits_on_the_service'] = df['limits_on_the_service'].apply(
            lambda v: '' if str(v).strip() == ':' else v
        )

    return df

df_final_clean_v2 = final_layer_clean(df_final_clean)
print("Done.")
df_final_clean_v2


Done.


,document_id,proposed_effective_date,approved_effective_date,service_name,service_type,renewal_or_new_or_replacement,limits_on_the_service,service_self_directed,service_providermanaged,serviceprovider_lrp,...,hcbs_taxonomy_1,hcbs_taxonomy_1a,hcbs_taxonomy_2,hcbs_taxonomy_2a,service_definition,taxonomy_code,taxonomy_category,taxonomy_service,limit_categories,limit_type_primary
0,AK0260R0600,7/1/21,07/01/21,Care Coordination,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,01 Case Management,01010 case management,NaN,NaN,Care Coordinators assist individuals to gain a...,1,Case Management,Case Management,,
1,AK0260R0600,7/1/21,07/01/21,Day Habilitation,Statutory Service,Service is included in approved waiver. There ...,"Per 7 AAC 130.260, the department will only pa...",0,1,0,...,04 Day Services,04020 day habilitation,NaN,NaN,Day habilitation services may be provided to a...,4,Day Services,Day Habilitation,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
2,AK0260R0600,7/1/21,07/01/21,Employment Services,Statutory Service,Service is included in approved waiver. There ...,NaN,0,0,0,...,03 Supported Employment,"03021 ongoing supported employment, individual",03 Supported Employment,"03022 ongoing supported employment, group",Employment services assist recipients to acqui...,3,Supported Employment,Supported Employment - Unspecified,,
3,AK0260R0600,7/1/21,07/01/21,Environmental Modifications,Other Service,Service is included in approved waiver. There ...,The State will pay for environmental modificat...,0,1,0,...,NaN,NaN,NaN,NaN,Environmental modification services may be pro...,14,"Equipment, Technology, and Modifications",Home/Vehicle Accessibility Adaptations,"Frequency / Quantity Caps , Cost / Budget Limi...",Cost / Budget Limits
4,AK0260R0600,7/1/21,07/01/21,Intensive Active Treatment,Other Service,Service is included in approved waiver. There ...,NaN,0,1,0,...,NaN,NaN,NaN,NaN,Intensive Active Treatment (IAT) assists parti...,4,Day Services,Community Integration,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19225,WY1061R0200,04/01/24,04/01/24,Special Family Habilitation Home,Other Service,Service is included in approved waiver. There ...,This service is not open to new participants w...,0,1,0,...,02 Round-the-Clock Services,02031 in-home residential habilitation,NaN,NaN,Special family habilitation home (SFHH) servic...,2,Round-the-Clock Services,Special Family Habilitation Home,"Location / Setting Restrictions , Prohibition ...",Location / Setting Restrictions
19226,WY1061R0200,04/01/24,04/01/24,Specialized Equipment,Other Service,Service is included in approved waiver. There ...,"Specialized Equipment has a $4,000 annual limi...",0,1,0,...,"14 Equipment, Technology, and Modifications",14031 equipment and technology,NaN,NaN,"Specialized Equipment includes: - Devices, con...",14,"Equipment, Technology, and Modifications",Equipment and Technology,"Hours / Time Restrictions , Frequency / Quanti...",Cost / Budget Limits
19227,WY1061R0200,04/01/24,04/01/24,"Speech, Hearing and Language Services",Extended State Plan Service,Service is included in approved waiver. There ...,"Speech, Hearing, and Language Services are ava...",0,1,0,...,11 Other Health and Therapeutic Services,"11100 speech, hearing, and language therapy",NaN,NaN,"Speech, Hearing, and Language Services consist...",11,Other Health and Therapeutic Services,"Speech, Hearing and Language Services","Frequency / Quantity Caps , Documentation Requ...",Extraction Artifact
19228,WY1061R0200,04/01/24,04/01/24,Supported Employment,Statutory Service,Service is included in approved waiver. There ...,Documentation that the service is not availabl...,1,1,0,...,03 Supported Employment,03010 job development,03 Supported Employment,"03021 ongoing supported employment, individual",Supported Employment Services are intended to ...,3,Supported Employment,Ongoing Supported Employment - Individua

In [65]:
# ── 3. Save ───────────────────────────────────────────────────────────────────
df_final_clean_v2.to_csv("../output/merged_with_corpus_service_level_categorized.csv", index=False, quoting=csv.QUOTE_ALL)
print(f"\nSaved {df_final_clean_v2.shape[0]:,} rows x {df_final_clean_v2.shape[1]} cols -> ../output/merged_with_corpus_service_level_categorized.csv")



Saved 19,230 rows x 34 cols -> ../output/merged_with_corpus_service_level_categorized.csv


In [66]:
import csv
import numpy as np
import pandas as pd
from pandas.api.types import is_bool_dtype, is_integer_dtype, is_object_dtype, is_string_dtype

def _is_stata_string_missing(value):
    if value is None or value is pd.NA:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False

def prepare_for_stata_export(df):
    df_stata = df.copy()

    for col in df_stata.columns:
        series = df_stata[col]
        is_text_col = (
            is_object_dtype(series.dtype)
            or is_string_dtype(series.dtype)
            or isinstance(series.dtype, pd.CategoricalDtype)
        )

        if is_text_col:
            # Stata string missing value is an empty string.
            df_stata[col] = series.map(
                lambda v: "" if _is_stata_string_missing(v) else str(v)
            )
        elif is_bool_dtype(series.dtype) or is_integer_dtype(series.dtype):
            df_stata[col] = pd.to_numeric(series, errors="coerce").astype("float64")
        else:
            df_stata[col] = series.mask(series.isna(), np.nan)

    return df_stata

df_export = df_final_clean_v2.copy()
df_stata = prepare_for_stata_export(df_export)

df_stata = df_stata.rename(columns={
    "provision_of_personal_care_description": "provision_of_personal_care_descr",
})

df_stata.to_stata("../output/1915c-service-level-v2.dta", write_index=False, version=118)
df_export.to_csv("../output/1915c-service-level-v2.csv", index=False, quoting=csv.QUOTE_ALL)

print(f"Saved {df_export.shape[0]:,} rows x {df_export.shape[1]} cols.")


Saved 19,230 rows x 34 cols.


In [67]:
df_export.columns

Index(['document_id', 'proposed_effective_date', 'approved_effective_date',
       'service_name', 'service_type', 'renewal_or_new_or_replacement',
       'limits_on_the_service', 'service_self_directed',
       'service_providermanaged', 'serviceprovider_lrp',
       'serviceprovider_relative', 'serviceprovider_lg',
       'provision_of_personal_care', 'provision_of_personal_care_description',
       'other_state_policies', 'other_state_policies_description',
       'waive_statewideness', 'geographic_limitations',
       'limited_implementation', 'year_1_participants', 'year_2_participants',
       'year_3_participants', 'year_4_participants', 'year_5_participants',
       'hcbs_taxonomy_1', 'hcbs_taxonomy_1a', 'hcbs_taxonomy_2',
       'hcbs_taxonomy_2a', 'service_definition', 'taxonomy_code',
       'taxonomy_category', 'taxonomy_service', 'limit_categories',
       'limit_type_primary'],
      dtype='object')